In [7]:
pip install pandas numpy faker

In [8]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('en_GB')
random.seed(42)
np.random.seed(42)

# --- CONFIG ---
NUM_SESSIONS = 2000
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 6, 30)

# GOV.UK style pages - mimicking a passport renewal journey
PAGES = [
    '/apply-renew-passport',
    '/apply-renew-passport/eligibility',
    '/apply-renew-passport/photo',
    '/apply-renew-passport/personal-details',
    '/apply-renew-passport/check-answers',
    '/apply-renew-passport/payment',
    '/apply-renew-passport/confirmation',
]

TRAFFIC_SOURCES = {
    'organic_search': 0.55,
    'direct': 0.20,
    'referral': 0.10,
    'paid_search': 0.10,
    'social': 0.05,
}

DEVICES = {
    'mobile': 0.52,
    'desktop': 0.38,
    'tablet': 0.10,
}

# --- BUILD SESSIONS ---
rows = []

for session_id in range(1, NUM_SESSIONS + 1):

    # Random session date/time
    session_start = START_DATE + timedelta(
        seconds=random.randint(0, int((END_DATE - START_DATE).total_seconds()))
    )

    # Pick device and source
    device = random.choices(
        list(DEVICES.keys()), weights=list(DEVICES.values())
    )[0]

    source = random.choices(
        list(TRAFFIC_SOURCES.keys()), weights=list(TRAFFIC_SOURCES.values())
    )[0]

    user_id = fake.uuid4()

    # Simulate how far through the journey the user gets
    # Mobile users drop off more. Paid search users complete more.
    if device == 'mobile':
        max_depth = random.choices(range(1, 8), weights=[30,20,18,14,10,5,3])[0]
    elif source == 'paid_search':
        max_depth = random.choices(range(1, 8), weights=[5,8,10,15,20,22,20])[0]
    else:
        max_depth = random.choices(range(1, 8), weights=[15,15,15,15,15,13,12])[0]

    # Create one row per page viewed in this session
    for depth, page in enumerate(PAGES[:max_depth], start=1):

        time_on_page = random.randint(10, 240) if depth < max_depth else random.randint(5, 60)

        rows.append({
            'session_id': f'sess_{session_id:04d}',
            'user_id': user_id,
            'event_timestamp': session_start + timedelta(seconds=sum(range(depth)) * 45),
            'event_name': 'page_view',
            'page_path': page,
            'page_depth': depth,
            'device_category': device,
            'traffic_source': source,
            'time_on_page_seconds': time_on_page,
            'country': random.choices(['United Kingdom','Ireland','United States','Australia'], weights=[88,4,4,4])[0],
        })

        # Add a form_submit event on the payment page
        if page == '/apply-renew-passport/payment' and random.random() > 0.15:
            rows.append({
                'session_id': f'sess_{session_id:04d}',
                'user_id': user_id,
                'event_timestamp': session_start + timedelta(seconds=sum(range(depth)) * 45 + 90),
                'event_name': 'form_submit',
                'page_path': page,
                'page_depth': depth,
                'device_category': device,
                'traffic_source': source,
                'time_on_page_seconds': 90,
                'country': random.choices(['United Kingdom','Ireland','United States','Australia'], weights=[88,4,4,4])[0],
            })

df = pd.DataFrame(rows)

print(f"✅ Dataset created: {len(df)} rows across {NUM_SESSIONS} sessions")
print(f"\nEvent types:\n{df['event_name'].value_counts()}")
print(f"\nDevice split:\n{df['device_category'].value_counts()}")
print(f"\nFirst 5 rows:")
df.head()

✅ Dataset created: 7016 rows across 2000 sessions

Event types:
event_name
page_view      6720
form_submit     296
Name: count, dtype: int64

Device split:
device_category
desktop    3136
mobile     2964
tablet      916
Name: count, dtype: int64

First 5 rows:


,session_id,user_id,event_timestamp,event_name,page_path,page_depth,device_category,traffic_source,time_on_page_seconds,country
0,sess_0001,d88ec136-4be0-436d-8782-f4da36ec71fb,2024-05-04 03:56:41,page_view,/apply-renew-passport,1,mobile,direct,13,United Kingdom
1,sess_0002,1be30069-25dd-434a-baa5-d7e2e62b3780,2024-05-11 09:38:53,page_view,/apply-renew-passport,1,desktop,organic_search,18,United Kingdom
2,sess_0002,1be30069-25dd-434a-baa5-d7e2e62b3780,2024-05-11 09:39:38,page_view,/apply-renew-passport/eligibility,2,desktop,organic_search,65,United Kingdom
3,sess_0002,1be30069-25dd-434a-baa5-d7e2e62b3780,2024-05-11 09:41:08,page_view,/apply-renew-passport/photo,3,desktop,organic_search,164,United Kingdom
4,sess_0002,1be30069-25dd-434a-baa5-d7e2e62b3780,2024-05-11 09:43:23,page_view,/apply-renew-passport/personal-details,4,desktop,organic_search,17,United Kingdom


In [9]:
# ── STEP 2: KPI PERFORMANCE FRAMEWORK ──────────────────────────────────────

# ── KPI 1: Task Completion Rate ────────────────────────────────────────────
# "Completed" = reached the confirmation page

sessions_started = df['session_id'].nunique()

sessions_completed = df[
    df['page_path'] == '/apply-renew-passport/confirmation'
]['session_id'].nunique()

task_completion_rate = (sessions_completed / sessions_started) * 100

print("=" * 50)
print("KPI 1 — TASK COMPLETION RATE")
print("=" * 50)
print(f"Sessions started:   {sessions_started:,}")
print(f"Sessions completed: {sessions_completed:,}")
print(f"Completion rate:    {task_completion_rate:.1f}%")
print()

# ── KPI 2: Funnel Drop-off by Page ─────────────────────────────────────────
# How many unique sessions reached each page?

print("=" * 50)
print("KPI 2 — FUNNEL DROP-OFF BY PAGE")
print("=" * 50)

funnel = []

for page in PAGES:
    sessions_at_page = df[df['page_path'] == page]['session_id'].nunique()
    pct_of_total = (sessions_at_page / sessions_started) * 100
    funnel.append({
        'page': page,
        'sessions': sessions_at_page,
        'pct_reached': round(pct_of_total, 1)
    })

funnel_df = pd.DataFrame(funnel)

# Drop-off = how many left at this page vs the one before
funnel_df['sessions_lost'] = funnel_df['sessions'].diff().fillna(0).abs().astype(int)
funnel_df['drop_off_pct'] = (
    funnel_df['sessions_lost'] / funnel_df['sessions'].shift(1) * 100
).fillna(0).round(1)

print(funnel_df[['page','sessions','pct_reached','drop_off_pct']].to_string(index=False))
print()

# ── KPI 3: Average Journey Time ────────────────────────────────────────────
# Only for sessions that completed the full journey

completed_sessions = df[
    df['page_path'] == '/apply-renew-passport/confirmation'
]['session_id'].unique()

completed_df = df[df['session_id'].isin(completed_sessions)]

journey_times = (
    completed_df.groupby('session_id')['time_on_page_seconds']
    .sum()
)

avg_journey_seconds = journey_times.mean()
avg_journey_minutes = avg_journey_seconds / 60

print("=" * 50)
print("KPI 3 — AVERAGE JOURNEY TIME (completers only)")
print("=" * 50)
print(f"Average journey time: {avg_journey_minutes:.1f} minutes")
print(f"Fastest 10%:          {journey_times.quantile(0.10)/60:.1f} minutes")
print(f"Slowest 10%:          {journey_times.quantile(0.90)/60:.1f} minutes")
print()

# ── KPI 4: Completion Rate by Device ───────────────────────────────────────
# Tells us if mobile users struggle more — a common GOV.UK finding

print("=" * 50)
print("KPI 4 — COMPLETION RATE BY DEVICE")
print("=" * 50)

for device in ['mobile', 'desktop', 'tablet']:
    device_sessions = df[df['device_category'] == device]['session_id'].nunique()
    device_completed = df[
        (df['device_category'] == device) &
        (df['page_path'] == '/apply-renew-passport/confirmation')
    ]['session_id'].nunique()
    rate = (device_completed / device_sessions * 100) if device_sessions > 0 else 0
    print(f"{device:<10}  started: {device_sessions:>4}   completed: {device_completed:>4}   rate: {rate:.1f}%")

KPI 1 — TASK COMPLETION RATE
Sessions started:   2,000
Sessions completed: 149
Completion rate:    7.4%

KPI 2 — FUNNEL DROP-OFF BY PAGE
                                  page  sessions  pct_reached  drop_off_pct
                 /apply-renew-passport      2000        100.0           0.0
     /apply-renew-passport/eligibility      1563         78.1          21.8
           /apply-renew-passport/photo      1220         61.0          21.9
/apply-renew-passport/personal-details       864         43.2          29.2
   /apply-renew-passport/check-answers       589         29.4          31.8
         /apply-renew-passport/payment       335         16.8          43.1
    /apply-renew-passport/confirmation       149          7.4          55.5

KPI 3 — AVERAGE JOURNEY TIME (completers only)
Average journey time: 14.1 minutes
Fastest 10%:          10.8 minutes
Slowest 10%:          17.3 minutes

KPI 4 — COMPLETION RATE BY DEVICE
mobile      started: 1021   completed:   35   rate: 3.4%
desktop   

In [10]:
# ── STEP 3: SQL ANALYSIS (BigQuery style) ──────────────────────────────────

import sqlite3

# Load our dataframe into a SQL table
conn = sqlite3.connect(':memory:')
df.to_sql('ga4_events', conn, index=False, if_exists='replace')

print("✅ GA4 events loaded into SQL. Table: ga4_events")
print(f"   Columns: {', '.join(df.columns.tolist())}")

✅ GA4 events loaded into SQL. Table: ga4_events
   Columns: session_id, user_id, event_timestamp, event_name, page_path, page_depth, device_category, traffic_source, time_on_page_seconds, country


In [11]:
# ── QUERY 1: Funnel Analysis ────────────────────────────────────────────────
# Identical structure to a BigQuery funnel query on GA4 exports

query_funnel = """
SELECT
    page_path,
    COUNT(DISTINCT session_id)              AS sessions,
    ROUND(
        COUNT(DISTINCT session_id) * 100.0
        / MAX(COUNT(DISTINCT session_id)) OVER (), 1
    )                                       AS pct_of_start
FROM
    ga4_events
WHERE
    event_name = 'page_view'
GROUP BY
    page_path
ORDER BY
    MIN(page_depth)
"""

funnel_sql = pd.read_sql_query(query_funnel, conn)
print("=" * 55)
print("QUERY 1 — Funnel: sessions reaching each page")
print("=" * 55)
print(funnel_sql.to_string(index=False))
print()

# ── QUERY 2: Drop-off by Device ─────────────────────────────────────────────
# Which device type struggles most at each stage?

query_device = """
SELECT
    device_category,
    page_path,
    COUNT(DISTINCT session_id)              AS sessions
FROM
    ga4_events
WHERE
    event_name = 'page_view'
GROUP BY
    device_category,
    page_path
ORDER BY
    device_category,
    MIN(page_depth)
"""

device_sql = pd.read_sql_query(query_device, conn)
print("=" * 55)
print("QUERY 2 — Sessions per page, broken down by device")
print("=" * 55)
print(device_sql.to_string(index=False))
print()

# ── QUERY 3: Traffic Source Quality ─────────────────────────────────────────
# Which source brings users most likely to complete?

query_source = """
SELECT
    traffic_source,
    COUNT(DISTINCT session_id)              AS total_sessions,
    COUNT(DISTINCT CASE
        WHEN page_path = '/apply-renew-passport/confirmation'
        THEN session_id END)                AS completions,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN page_path = '/apply-renew-passport/confirmation'
            THEN session_id END) * 100.0
        / COUNT(DISTINCT session_id), 1
    )                                       AS completion_rate_pct
FROM
    ga4_events
GROUP BY
    traffic_source
ORDER BY
    completion_rate_pct DESC
"""

source_sql = pd.read_sql_query(query_source, conn)
print("=" * 55)
print("QUERY 3 — Completion rate by traffic source")
print("=" * 55)
print(source_sql.to_string(index=False))
print()

# ── QUERY 4: Slowest pages by avg time ──────────────────────────────────────
# Where are users spending the most time? Could signal confusion

query_time = """
SELECT
    page_path,
    ROUND(AVG(time_on_page_seconds), 1)     AS avg_seconds,
    ROUND(AVG(time_on_page_seconds)/60, 2)  AS avg_minutes,
    COUNT(DISTINCT session_id)              AS sessions
FROM
    ga4_events
WHERE
    event_name = 'page_view'
GROUP BY
    page_path
ORDER BY
    avg_seconds DESC
"""

time_sql = pd.read_sql_query(query_time, conn)
print("=" * 55)
print("QUERY 4 — Average time spent on each page")
print("=" * 55)
print(time_sql.to_string(index=False))

QUERY 1 — Funnel: sessions reaching each page
                             page_path  sessions  pct_of_start
                 /apply-renew-passport      2000         100.0
     /apply-renew-passport/eligibility      1563          78.2
           /apply-renew-passport/photo      1220          61.0
/apply-renew-passport/personal-details       864          43.2
   /apply-renew-passport/check-answers       589          29.5
         /apply-renew-passport/payment       335          16.8
    /apply-renew-passport/confirmation       149           7.5

QUERY 2 — Sessions per page, broken down by device
device_category                              page_path  sessions
        desktop                  /apply-renew-passport       767
        desktop      /apply-renew-passport/eligibility       654
        desktop            /apply-renew-passport/photo       549
        desktop /apply-renew-passport/personal-details       425
        desktop    /apply-renew-passport/check-answers       315
        

In [12]:
# ── STEP 4: DATA QUALITY AND VALIDATION ────────────────────────────────────

issues_found = []

print("=" * 55)
print("DATA QUALITY REPORT — GOV.UK Passport Renewal Journey")
print("=" * 55)
print()

# ── CHECK 1: Missing values ─────────────────────────────────────────────────
print("CHECK 1 — Missing values")
print("-" * 40)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

if missing.sum() == 0:
    print("✅ No missing values found across all columns")
else:
    print("⚠️  Missing values detected:")
    print(missing_report[missing_report['missing_count'] > 0])
    issues_found.append("Missing values present")
print()

# ── CHECK 2: Duplicate events ───────────────────────────────────────────────
print("CHECK 2 — Duplicate events")
print("-" * 40)

dupes = df.duplicated(subset=['session_id','event_name','page_path','event_timestamp']).sum()

if dupes == 0:
    print("✅ No duplicate events found")
else:
    print(f"⚠️  {dupes} duplicate events detected — would inflate pageview counts")
    issues_found.append(f"{dupes} duplicate events")
print()

# ── CHECK 3: Invalid page paths ─────────────────────────────────────────────
print("CHECK 3 — Page paths outside expected journey")
print("-" * 40)

unexpected_pages = df[~df['page_path'].isin(PAGES)]

if len(unexpected_pages) == 0:
    print("✅ All page paths match the expected journey")
else:
    print(f"⚠️  {len(unexpected_pages)} events on unexpected pages:")
    print(unexpected_pages['page_path'].value_counts())
    issues_found.append("Unexpected page paths found")
print()

# ── CHECK 4: Journey order violations ───────────────────────────────────────
# Users should not skip pages — e.g. jumping from page 1 to page 5
print("CHECK 4 — Journey order violations (page skipping)")
print("-" * 40)

page_order = {page: i for i, page in enumerate(PAGES)}
df['page_order_num'] = df['page_path'].map(page_order)

session_pages = df[df['event_name'] == 'page_view'].groupby('session_id').apply(
    lambda x: x.sort_values('event_timestamp')['page_order_num'].tolist()
)

violations = 0
for session, pages in session_pages.items():
    for i in range(1, len(pages)):
        if pages[i] < pages[i-1]:
            violations += 1
            break

if violations == 0:
    print("✅ No journey order violations found")
else:
    print(f"⚠️  {violations} sessions contain out-of-order page visits")
    issues_found.append(f"{violations} journey order violations")
print()

# ── CHECK 5: Timestamp sanity ───────────────────────────────────────────────
print("CHECK 5 — Timestamps within expected date range")
print("-" * 40)

df['event_timestamp'] = pd.to_datetime(df['event_timestamp'])
out_of_range = df[
    (df['event_timestamp'] < START_DATE) |
    (df['event_timestamp'] > END_DATE)
]

if len(out_of_range) == 0:
    print(f"✅ All timestamps fall within {START_DATE.date()} to {END_DATE.date()}")
else:
    print(f"⚠️  {len(out_of_range)} events outside the expected date range")
    issues_found.append("Out-of-range timestamps")
print()

# ── CHECK 6: Device and source values ───────────────────────────────────────
print("CHECK 6 — Unexpected device or traffic source values")
print("-" * 40)

valid_devices  = {'mobile', 'desktop', 'tablet'}
valid_sources  = set(TRAFFIC_SOURCES.keys())

bad_devices = df[~df['device_category'].isin(valid_devices)]['device_category'].unique()
bad_sources = df[~df['traffic_source'].isin(valid_sources)]['traffic_source'].unique()

if len(bad_devices) == 0:
    print("✅ All device categories are valid")
else:
    print(f"⚠️  Unexpected device values: {bad_devices}")
    issues_found.append("Invalid device values")

if len(bad_sources) == 0:
    print("✅ All traffic sources are valid")
else:
    print(f"⚠️  Unexpected source values: {bad_sources}")
    issues_found.append("Invalid traffic sources")
print()

# ── CHECK 7: Session event volume sanity ────────────────────────────────────
print("CHECK 7 — Suspiciously high event counts per session (bot detection)")
print("-" * 40)

events_per_session = df.groupby('session_id').size()
high_volume = events_per_session[events_per_session > 20]

if len(high_volume) == 0:
    print("✅ No sessions with suspiciously high event counts")
else:
    print(f"⚠️  {len(high_volume)} sessions with >20 events — possible bot traffic:")
    print(high_volume.sort_values(ascending=False).head(5))
    issues_found.append(f"{len(high_volume)} potential bot sessions")
print()

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("=" * 55)
print("SUMMARY")
print("=" * 55)

if len(issues_found) == 0:
    print("✅ All checks passed — data is clean and ready for reporting")
else:
    print(f"⚠️  {len(issues_found)} issue(s) require afttention before publishing:")
    for issue in issues_found:
        print(f"   • {issue}")

DATA QUALITY REPORT — GOV.UK Passport Renewal Journey

CHECK 1 — Missing values
----------------------------------------
✅ No missing values found across all columns

CHECK 2 — Duplicate events
----------------------------------------
✅ No duplicate events found

CHECK 3 — Page paths outside expected journey
----------------------------------------
✅ All page paths match the expected journey

CHECK 4 — Journey order violations (page skipping)
----------------------------------------
✅ No journey order violations found

CHECK 5 — Timestamps within expected date range
----------------------------------------
✅ All timestamps fall within 2024-01-01 to 2024-06-30

CHECK 6 — Unexpected device or traffic source values
----------------------------------------
✅ All device categories are valid
✅ All traffic sources are valid

CHECK 7 — Suspiciously high event counts per session (bot detection)
----------------------------------------
✅ No sessions with suspiciously high event counts

SUMMARY
✅

/tmp/ipykernel_6005/2112489814.py:64: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  session_pages = df[df['event_name'] == 'page_view'].groupby('session_id').apply(


In [14]:
# Recreate weekly sessions (skipped the matplotlib step)
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'])
df['week'] = df['event_timestamp'].dt.to_period('W').apply(lambda r: r.start_time)

weekly = (
    df[df['event_name'] == 'page_view']
    .groupby('week')['session_id']
    .nunique()
    .reset_index()
)

print("✅ Weekly sessions recreated")

✅ Weekly sessions recreated


In [15]:
# ── STEP 5: EXPORT DATA FOR LOOKER STUDIO ───────────────────────────────────

# Export 1: Raw events table
df.to_csv('ga4_events.csv', index=False)

# Export 2: Funnel summary (ready to chart immediately in Looker)
funnel_export = []

for page in PAGES:
    sessions_at_page = df[df['page_path'] == page]['session_id'].nunique()
    pct_of_total     = round(sessions_at_page / sessions_started * 100, 1)
    funnel_export.append({
        'page_path':       page,
        'page_label':      page.split('/')[-1].replace('-', ' ').title(),
        'sessions':        sessions_at_page,
        'pct_reached':     pct_of_total,
    })

funnel_export_df = pd.DataFrame(funnel_export)
funnel_export_df.to_csv('funnel_summary.csv', index=False)

# Export 3: Completion by device
device_export = []

for device in ['mobile', 'desktop', 'tablet']:
    started   = df[df['device_category'] == device]['session_id'].nunique()
    completed = df[
        (df['device_category'] == device) &
        (df['page_path'] == '/apply-renew-passport/confirmation')
    ]['session_id'].nunique()
    rate = round(completed / started * 100, 1) if started else 0
    device_export.append({
        'device':           device,
        'sessions_started': started,
        'completions':      completed,
        'completion_rate':  rate,
    })

device_export_df = pd.DataFrame(device_export)
device_export_df.to_csv('completion_by_device.csv', index=False)

# Export 4: Completion by traffic source
source_export = []

for source in TRAFFIC_SOURCES.keys():
    started   = df[df['traffic_source'] == source]['session_id'].nunique()
    completed = df[
        (df['traffic_source'] == source) &
        (df['page_path'] == '/apply-renew-passport/confirmation')
    ]['session_id'].nunique()
    rate = round(completed / started * 100, 1) if started else 0
    source_export.append({
        'traffic_source':   source,
        'sessions_started': started,
        'completions':      completed,
        'completion_rate':  rate,
    })

source_export_df = pd.DataFrame(source_export)
source_export_df.to_csv('completion_by_source.csv', index=False)

# Export 5: Weekly sessions
weekly_export = weekly.copy()
weekly_export.columns = ['week_start', 'sessions']
weekly_export['week_start'] = weekly_export['week_start'].astype(str)
weekly_export.to_csv('weekly_sessions.csv', index=False)

print("✅ Exported 5 files:")
print("   • ga4_events.csv          — full raw event data")
print("   • funnel_summary.csv      — funnel drop-off by page")
print("   • completion_by_device.csv — completion rate by device")
print("   • completion_by_source.csv — completion rate by traffic source")
print("   • weekly_sessions.csv      — session volume over time")

✅ Exported 5 files:
   • ga4_events.csv          — full raw event data
   • funnel_summary.csv      — funnel drop-off by page
   • completion_by_device.csv — completion rate by device
   • completion_by_source.csv — completion rate by traffic source
   • weekly_sessions.csv      — session volume over time
